# Imports & env load

In [1]:
import chromadb
from langchain_chroma import Chroma
from groq import Groq
from dotenv import load_dotenv
import os 

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
client = Groq()

## Embedding Model

In [2]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

C:\Users\Vaibhav Kesarwani\AppData\Local\Temp\ipykernel_20216\726496232.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
c:\Users\Vaibhav Kesarwani\Desktop\AI_Training_Batch_May_2026\submissions\assignments\assignment-02\vaibhav-kesarwani\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Model Selection

In [3]:
model_name = 'openai/gpt-oss-120b'

# Query Expansion

## Vector Database Connection

In [4]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [5]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'

In [6]:
vectorstore_persisted = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [7]:
chromadb_client.list_collections()

['hypothetical_questions', 'tesla-10k-2019-to-2023']

## Vector database retrival

In [8]:
retriever = vectorstore_persisted.as_retriever(
    search_type="similarity",
    search_kwargs={
        'k': 3
    }
)

## Prompt for Query expansion

In [9]:
query_expansion_system_message = """
You are an financial domain expert assisting in answering questions related to 10-k reports.
Perform query expansion on the question below. If there are multiple common ways of phrasing a user question \
or common synonyms for key words in the question, make sure to return multiple versions \
of the query with the different phrasings.

If there are acronyms or words you are not familiar with, do not try to rephrase them.

Return at least 3 versions of the question as a list.
Generate only a list of questions, each question in a new line.
Do not number the list of questions or use bullet points.
Do not mention anything before or after the list.
"""

user_message_template="""
<Question>
{question}
</Question>
"""

In [11]:
user_query = "What was the automotive revenue in 2021?"

In [12]:
prompt = [
    {"role" : "system", "content" : query_expansion_system_message},
    {"role" : "user", "content" : user_message_template.format(question=user_query)}
]

In [13]:
response = client.chat.completions.create(
    model=model_name,
    messages=prompt,
    temperature=0.2
)

query_expansion_list = response.choices[0].message.content.strip().split("\n")

In [14]:
query_expansion_list

['What was the automotive revenue in 2021?',
 'How much automotive revenue did the company generate in 2021?',
 'What were the 2021 revenues from the automotive segment?']

In [15]:
expanded_context_list = []

for query in query_expansion_list:
    expanded_context_list.extend([d.page_content for d in retriever.invoke(query)])

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [16]:
len(expanded_context_list)

9

In [17]:
expanded_context_list

['2022\n2021\n$\n%\n$\n%\nAutomotive\tsales\n$\n78,509\t\n$\n67,210\t\n$\n44,125\t\n$\n11,299\t\n17\t\n%\n$\n23,085\t\n52\t\n%\nAutomotive\tregulatory\tcredits\n1,790\t\n1,776\t\n1,465\t\n14\t\n1\t\n%\n311\t\n21\t\n%\nAutomotive\tleasing\n2,120\t\n2,476\t\n1,642\t\n(356)\n(14)\n%\n834\t\n51\t\n%\nTotal\tautomotive\trevenues\n82,419\t\n71,462\t\n47,232\t\n10,957\t\n15\t\n%\n24,230\t\n51\t\n%\nServices\tand\tother\n8,319\t\n6,091\t\n3,802\t\n2,228\t\n37\t\n%\n2,289\t\n60\t\n%\nTotal\tautomotive\t&\tservices\tand\tother\tsegment\nrevenue\n90,738\t\n77,553\t\n51,034\t\n13,185\t\n17\t\n%\n26,519\t\n52\t\n%\nEnergy\tgeneration\tand\tstorage\tsegment\trevenue\n6,035\t\n3,909\t\n2,789\t\n2,126\t\n54\t\n%\n1,120\t\n40\t\n%\nTotal\trevenues\n$\n96,773\t\n$\n81,462\t\n$\n53,823\t\n$\n15,311\t\n19\t\n%\n$\n27,639\t\n51\t\n%\nAutomotive\t&\tServices\tand\tOther\tSegment\nAutomotive\tsales\trevenue\tincludes\trevenues\trelated\tto\tcash\tand\tfinancing\tdeliveries\tof\tnew\tModel\tS,\tModel\tX,\tSem

In [19]:
final_context_documents = set(expanded_context_list)

len(final_context_documents)

4

In [20]:
final_context_documents

{'(Dollars\tin\tmillions)\n2023\n2022\n2021\n$\n%\n$\n%\nCost\tof\trevenues\nAutomotive\tsales\n$\n65,121\t\n$\n49,599\t\n$\n32,415\t\n$\n15,522\t\n31\t\n%\n$\n17,184\t\n53\t\n%\nAutomotive\tleasing\n1,268\t\n1,509\t\n978\t\n(241)\n(16)\n%\n531\t\n54\t\n%\nTotal\tautomotive\tcost\tof\trevenues\n66,389\t\n51,108\t\n33,393\t\n15,281\t\n30\t\n%\n17,715\t\n53\t\n%\nServices\tand\tother\n7,830\t\n5,880\t\n3,906\t\n1,950\t\n33\t\n%\n1,974\t\n51\t\n%\nTotal\tautomotive\t&\tservices\tand\tother\tsegment\ncost\tof\trevenues\n74,219\t\n56,988\t\n37,299\t\n17,231\t\n30\t\n%\n19,689\t\n53\t\n%\nEnergy\tgeneration\tand\tstorage\tsegment\n4,894\t\n3,621\t\n2,918\t\n1,273\t\n35\t\n%\n703\t\n24\t\n%\nTotal\tcost\tof\trevenues\n$\n79,113\t\n$\n60,609\t\n$\n40,217\t\n$\n18,504\t\n31\t\n%\n$\n20,392\t\n51\t\n%\nGross\tprofit\ttotal\tautomotive\n$\n16,030\t\n$\n20,354\t\n$\n13,839\t\nGross\tmargin\ttotal\tautomotive\n19.4\t\n%\n28.5\t\n%\n29.3\t\n%\nGross\tprofit\ttotal\tautomotive\t&\tservices\tand\tothe

## Final Query Expansion Result

In [21]:
final_query_system = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

In [22]:
final_query_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

In [23]:
context_for_query = "\n\n---\n\n".join(
    map(str, final_context_documents)
)

In [24]:
prompt = [
    {
        "role": "system",
        "content": final_query_system
    },
    {
        "role": "user",
        "content": final_query_user_message_template.format(
            context=context_for_query,
            question=user_query
        )
    }
]

In [25]:
try:
    response = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0
    )

    prediction = response.choices[0].message.content.strip()
except Exception as e:
    print(e)

print(prediction)

The automotive revenue in 2021 was **$47,232 million**.
